# Colab: M2 re-check with the longer settling stage

**Why this batch exists.** The diagnostics showed that letting the model settle for 5,000 steps (instead of 500) before S and T start learning fixes most of the realistic-case failure. That is now the default in `configs/pinn.yaml`. This batch re-runs the key synthetic cases with it, so the M2 report uses the settings we will actually use on real data.

**Before you start:**
1. Upload the new `pinn_code.zip` (from the laptop's `pinn` folder) to `MyDrive/pinn/`, replacing the old one. The code has changed.
2. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** 8 runs of about 17 minutes each, so about 2.5 hours. Keep the tab open.

**If Colab disconnects:** just do **Run all** again. Runs that already finished are skipped.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Build the simulated aquifers (about 30 seconds)

In [ ]:
!python scripts/make_synthetic.py

## Re-check runs (most important first)

| Run | Question it answers |
|---|---|
| run1, seeds 0, 1, 2 | does the gate (S, T, seasonal pumping within 10%) still pass, and on more seeds? |
| run3 | realistic case (30 wells, 4 readings a year): how close are S, T and total pumping now? |
| run3 + T fixed 30% wrong | the set-up planned for real data: T from the CGWB report, S and pumping learned |
| run9, run10 | case D, where the truth deliberately breaks the model's assumptions (and R is 20% low in run10) |
| run4 + data weight 1 | noisy data (0.3 m): still the open problem |

In [ ]:
RECHECK = [
    ("run1", "--seed 0", "v2_seed0"),
    ("run3", "", "v2"),
    ("run1", "--seed 1", "v2_seed1"),
    ("run3", "--fix-T 1950", "v2_fixT1950"),
    ("run9", "", "v2"),
    ("run10", "", "v2"),
    ("run1", "--seed 2", "v2_seed2"),
    ("run4", "--set training.weighting.init=1", "v2_w1"),
]
for run, extra, tag in RECHECK:
    if glob.glob(f"{DRIVE_ROOT}/runs/*_m2_{run}_{tag}/metrics.json"):
        print("already done:", run, tag)
        continue
    !python -u scripts/run_m2.py {run} {extra} --tag {tag} 2>&1 | tee -a {DRIVE_ROOT}/runs/logs/m2_v2.log

## Bundle the results for Claude
This makes `MyDrive/pinn/m2_v2_results.zip` (small, no model files). Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/m2_v2_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{DRIVE_ROOT}/runs/**/*", recursive=True):
        rel = os.path.relpath(path, f"{DRIVE_ROOT}/runs")
        if os.path.isfile(path) and not path.endswith(".pt") and ("_v2" in rel or rel.endswith("m2_v2.log")):
            z.write(path, rel)
print("wrote", out, f"({os.path.getsize(out) / 1e6:.2f} MB)")